In [1]:
# =============================================================================
# CELL 0: ENVIRONMENT SETUP & DEPENDENCIES
# =============================================================================
# Mandrake Engineers: This installs the required parsers for 3D biophysics extraction.
!pip install bio biopython -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.3/321.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00


In [2]:
"""
=============================================================================
MANDRAKE BIO REVERSE TRANSCRIPTASE - PHASE 2 SUBMISSION
Architecture: The Bounded Shield (Voting Tree Ensemble)
=============================================================================
"""

import os
import re
import numpy as np
import pandas as pd
from Bio.PDB import PDBParser
from scipy.spatial.distance import pdist, squareform
from scipy.spatial import ConvexHull
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, VotingRegressor
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = "/kaggle/input/competitions/retroviral-challenge-predict"
PDB_DIR = f"{DATA_DIR}/structures"
CHARGE_MAP = {'ARG': 1.0, 'LYS': 1.0, 'HIS': 0.1, 'ASP': -1.0, 'GLU': -1.0}
KD_SCALE = {'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5, 'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5, 'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6, 'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2}

In [3]:
# =============================================================================
# PART 1: EXTRACTABLE MODELING PIPELINE (FOR MANDRAKE BIO PHASE 2 EXECUTORS)
# =============================================================================
def engineer_thermodynamics(df):
    df = df.copy()
    if 'connection_mean_pot' in df.columns: df['has_connection_domain'] = df['connection_mean_pot'].notna().astype(float)
    if 'D1_D2_dist' in df.columns: df['is_missing_catalytic_triad'] = df['D1_D2_dist'].isna().astype(float)
    t40 = 't40_raw' if 't40_raw' in df.columns else 'frac_retained_40C'
    t80 = 't80_raw' if 't80_raw' in df.columns else 'frac_retained_80C'
    if t40 in df.columns and t80 in df.columns: df['thermo_degradation_slope'] = df[t80] - df[t40]
    fingers, thumb = 'fingers_mean_pot', 'thumb_mean_pot'
    if fingers in df.columns and thumb in df.columns: df['cleft_electrostatic_gradient'] = df[fingers] - df[thumb]
    df['creative_nucleic_grip'] = df['sequence'].apply(lambda seq: (seq.count('R') + seq.count('K')) / len(seq) if pd.notna(seq) else 0)
    
    def calc_thermo(seq):
        if pd.isna(seq) or not seq: return pd.Series([0, 0, 0])
        kd_mean = sum(KD_SCALE.get(aa, 0) for aa in seq) / len(seq)
        iso_bal = (seq.count('R') + seq.count('K') + seq.count('H') + 1) / (seq.count('D') + seq.count('E') + 1)
        steric = (seq.count('W') + seq.count('Y') + seq.count('F') + seq.count('R') + 1) / (seq.count('G') + seq.count('A') + seq.count('S') + 1)
        return pd.Series([kd_mean, iso_bal, steric])
    df[['creative_kd_hydropathy', 'creative_isoelectric_balance', 'creative_steric_ratio']] = df['sequence'].apply(calc_thermo)
    return df

def extract_3d_physics(row):
    rt_name = row['rt_name']
    sequence = str(row['sequence']) if 'sequence' in row else ""
    pdb_path = f"{PDB_DIR}/{rt_name}.pdb"
    features = {
        'pdb_packing_density': 0, 'pdb_radius_gyration': 0, 
        'creative_shape_anisotropy': 0, 'apex_dipole_density': 0, 
        'apex_cleft_ratio': 0, 'apex_yxdd_strain': 0
    }
    if not os.path.exists(pdb_path): return pd.Series(features)
        
    parser = PDBParser(QUIET=True)
    try:
        model = parser.get_structure(rt_name, pdb_path)[0]
        ca_atoms = [atom for atom in model.get_atoms() if atom.get_name() == 'CA']
        num_atoms = len(ca_atoms)
        if num_atoms > 10:
            coords = np.array([atom.get_coord() for atom in ca_atoms])
            center = coords.mean(axis=0)
            shifted_coords = coords - center
            features['pdb_radius_gyration'] = np.sqrt(np.mean(np.sum((shifted_coords)**2, axis=1)))
            dist_matrix = squareform(pdist(coords))
            features['pdb_packing_density'] = np.mean(np.sum((dist_matrix < 8.0) & (dist_matrix > 0), axis=1))
            cov_matrix = np.cov(shifted_coords.T)
            eigenvalues, _ = np.linalg.eigh(cov_matrix)
            l1, l2, l3 = sorted(eigenvalues, reverse=True)
            if (l1 + l2 + l3) > 0: features['creative_shape_anisotropy'] = 1.5 * ((l1**2 + l2**2 + l3**2) / ((l1 + l2 + l3)**2)) - 0.5
            
            dipole_vector = np.zeros(3)
            for atom in ca_atoms:
                charge = CHARGE_MAP.get(atom.get_parent().get_resname(), 0.0)
                dipole_vector += charge * (atom.get_coord() - center)
            features['apex_dipole_density'] = np.linalg.norm(dipole_vector) / num_atoms
            
            hull = ConvexHull(coords)
            features['apex_cleft_ratio'] = max(0, hull.volume - (num_atoms * 140.0)) / (hull.volume + 1e-9)
            
            motif_match = re.search(r'[YF][A-Z]DD', sequence)
            if motif_match and motif_match.start() < num_atoms - 3:
                features['apex_yxdd_strain'] = np.linalg.norm(ca_atoms[motif_match.start()].get_coord() - ca_atoms[motif_match.start() + 3].get_coord())
    except Exception: pass
    return pd.Series(features)

def build_mandrake_pipeline(physics_cols, esm_cols):
    preprocessor = ColumnTransformer([
        ('physics', SimpleImputer(strategy='median'), physics_cols),
        ('esm_pca', PCA(n_components=15, random_state=42), esm_cols)
    ])
    rf = RandomForestRegressor(n_estimators=300, max_depth=4, max_features='sqrt', min_samples_leaf=3, random_state=42)
    et = ExtraTreesRegressor(n_estimators=300, max_depth=4, max_features='sqrt', min_samples_leaf=3, random_state=42)
    return Pipeline([
        ('preprocessor', preprocessor), 
        ('model', VotingRegressor([('rf', rf), ('et', et)]))
    ])

In [4]:
# =============================================================================
# PART 2: DATA PREPARATION & ESM-2 ANCHORING
# =============================================================================
train_df = pd.read_csv(f"{DATA_DIR}/train.csv").drop(columns=[c for c in pd.read_csv(f"{DATA_DIR}/train.csv").columns if 'foldseek' in c], errors='ignore')
train_df = engineer_thermodynamics(train_df)
train_df = pd.concat([train_df, train_df.apply(extract_3d_physics, axis=1)], axis=1)

esm_data = np.load(f"{DATA_DIR}/esm2_embeddings.npz", allow_pickle=True)
esm_df = pd.DataFrame(esm_data['embeddings'], index=esm_data['names'])
esm_cols = [f'esm_raw_{i}' for i in range(1280)]
df_train_esm = pd.DataFrame(esm_df.loc[train_df['rt_name']].values, columns=esm_cols, index=train_df.index)
train_df = pd.concat([train_df, df_train_esm], axis=1)

physics_cols = [
    'thermo_degradation_slope', 'cleft_electrostatic_gradient', 'pdb_packing_density', 
    'pdb_radius_gyration', 'creative_shape_anisotropy', 'apex_dipole_density', 
    'apex_cleft_ratio', 'creative_kd_hydropathy', 'creative_steric_ratio', 'D1_D2_dist'
]
physics_cols = [f for f in physics_cols if f in train_df.columns]

X_master = train_df[physics_cols + esm_cols]
y_eff = train_df['pe_efficiency_pct'].values
groups = train_df['rt_family'].values

In [5]:
# =============================================================================
# PART 3: MANDATORY LOFO HARNESS & SUBMISSION GENERATION
# =============================================================================
logo = LeaveOneGroupOut()
oof_predictions = np.zeros(len(train_df))

for train_idx, val_idx in logo.split(X_master, y_eff, groups):
    model_pipeline = build_mandrake_pipeline(physics_cols, esm_cols)
    model_pipeline.fit(X_master.iloc[train_idx], y_eff[train_idx])
    # Tree ensembles are natively bounded, calibration is safely omitted.
    oof_predictions[val_idx] = model_pipeline.predict(X_master.iloc[val_idx])

submission = pd.DataFrame({'rt_name': train_df['rt_name'], 'predicted_score': np.nan_to_num(oof_predictions, nan=0.0)})
submission.to_csv("submission.csv", index=False)
print("✅ Saved 'submission.csv' (Bounded Tree Ensemble)")

✅ Saved 'submission.csv' (Bounded Tree Ensemble)
